In [ ]:
import sys

import numpy as np
from transforms3d.axangles import axangle2mat, mat2axangle
import torch
import plotly.graph_objects as go

from mano_pybullet.hand_model import HandModel20

In [ ]:
sys.path.append("..")

from utils.grasp_utils import get_handmodel
from model.hand_opt import AdamGraspTransfer

In [ ]:
def mat2rvec(mat):
    """Convert rotation matrix to rotation vector."""
    axis, angle = mat2axangle(mat, unit_thresh=1e-05)
    return axis * angle

In [ ]:
# NOTE: Set the mano hand models dir here. When using with a script, load this directory from a some config file

%env MANO_MODELS_DIR=/home/ninad/Projects/MANO/MANO_Hand_Model/mano_v1_2/models

In [ ]:
# Load data for the 00100 frame
data = np.load("../data/sample_hamer_output.npz", allow_pickle=True)

In [ ]:
for k in data.keys():
  print(k)

In [ ]:
is_left_hand = (data['right'][0] == 0)
print("Is left hand? -->", is_left_hand)

In [ ]:
mano_params = data['pred_mano_params'].item()

In [ ]:
type(mano_params)

In [ ]:
mano_params.keys()

In [ ]:
hand_rotn_mat = mano_params['global_orient'][0][0]
hand_theta_mat = mano_params['hand_pose'][0]
print(hand_rotn_mat.shape)
print(hand_theta_mat.shape)

In [ ]:
hand_theta_full = np.array([mat2rvec(hand_rotn_mat)] + [mat2rvec(hand_theta_mat[i]) for i in range(hand_theta_mat.shape[0])])
print(hand_theta_full.shape)

In [ ]:
hand_model = HandModel20(left_hand=is_left_hand)

In [ ]:
angles, palm_basis = hand_model.mano_to_angles(hand_theta_full)

In [ ]:
len(angles)

In [ ]:
angles

In [ ]:
palm_basis

In [ ]:
source_gripper = "mano_left" if is_left_hand else "mano_right"
target_gripper = "fetch_gripper"
device = "cpu"

In [ ]:
source_model = get_handmodel(
  source_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [ ]:
target_model = get_handmodel(
  target_gripper,
  1,
  device,
  json_path="urdf_assets_meta.json",
  datadir="../grippers/"
)

In [ ]:
# SOURCE GRIPPER (MANO) POSE + DOFS

grasp_pose = torch.zeros(9)
# grasp_pose[0:3] = torch.tensor([0.1, 0.2, 0.3])
# Identity rotation in 6d rot representation is: (1,0,0,0,1,0)
grasp_rotn_6d = palm_basis.T.reshape(-1)[:6]
grasp_pose[3:] = torch.tensor(grasp_rotn_6d)

print("Pose:", grasp_pose)

grasp_dofs = -1 * torch.tensor(angles)

print("DOFS:", grasp_dofs)

sample_grasp_q = (
  torch.cat(
    [
      grasp_pose,
      grasp_dofs,
    ]
  )
  .unsqueeze(0)
  .to(device)
  .float()
)

In [ ]:
source_model.dynamic_joints_q_lower.squeeze(0)

In [ ]:
source_model.dynamic_joints_q_upper.squeeze(0)

In [ ]:
grasp_dofs

In [ ]:
grasp_transfer_opt = AdamGraspTransfer(
  source_gripper,
  target_gripper,
  learning_rate=1e-3,
  device=device
)

In [ ]:
q_traj, energy, _ = grasp_transfer_opt.run_adam(
  sample_grasp_q.squeeze(0), running_name="test"
)

In [ ]:
print(q_traj.shape)
best_q = q_traj[0, -1]
print(best_q.shape)

In [ ]:
if best_q.shape[0] != 9 + len(target_model.dynamic_joints):
  # We optimized only for pose, so need to provide dummy joints
  best_q = torch.cat((best_q, target_model.dynamic_joints_q_lower[0]), dim=0)

In [ ]:
print("Plotting TARGET and SOURCE together...")

vis_data = source_model.get_plotly_data(q=sample_grasp_q, color='red')
vis_data += target_model.get_plotly_data(q=best_q.unsqueeze(0).float().to(device), color='green')
fig = go.Figure(data=vis_data)
fig.show()
# fig.write_html("../logs_viz/gtransfer_test.html")
